In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("powerplant_data.csv")

In [4]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [5]:
#AT => Average Temperature
#V => Vaccum
#AP =>Pressure
#RH => Humidity
#PE => Produced Energy

In [6]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [8]:
X=df.drop("PE",axis=1)
y=df["PE"]

In [10]:
y.head()

0    480.48
1    445.75
2    438.76
3    453.09
4    464.43
Name: PE, dtype: float64

In [13]:
#Split our data
from sklearn.model_selection import train_test_split
X_train,X_test, y_train, y_test=train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
df.shape

(9568, 5)

In [19]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

In [20]:
X_train_scaled

array([[ 0.74805289,  0.72006931, -0.32660017, -0.49711722],
       [ 0.86181948,  1.26515721, -0.98521113,  0.8181501 ],
       [ 0.93409473,  1.52314975,  0.32523844,  0.80167494],
       ...,
       [-0.22097078, -0.834965  ,  0.36756563, -0.83554456],
       [ 0.94747903,  1.14245344, -0.41971997, -0.45455637],
       [-1.77355014, -1.19049131,  1.92520594,  0.91837402]],
      shape=(7654, 4))

In [21]:
X_test_scaled

array([[ 1.36255441,  0.24305848, -1.27798491, -1.04586379],
       [ 0.82147733,  1.36986442, -0.74695303,  0.31071896],
       [-0.247116  , -0.73708132,  1.92304887, -0.14668594],
       ...,
       [-0.67970645, -1.15815506, -0.31652036, -0.05697206],
       [ 1.33136451,  1.34463153, -0.87558808, -0.39000089],
       [-0.26474508, -0.26711887,  0.33984824,  1.14397069]],
      shape=(1914, 4))

In [29]:
import torch
import torch.nn as nn

X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [25]:
type(X_test_scaled)

numpy.ndarray

In [26]:
type(y_train)

pandas.Series

In [27]:
y_train.shape

(7654,)

In [28]:
y_train

5487    442.75
3522    432.52
6916    428.80
7544    426.07
7600    436.58
         ...  
5734    436.44
5191    441.20
5390    464.26
860     440.45
7270    484.44
Name: PE, Length: 7654, dtype: float64

In [35]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

In [37]:
train_loader=DataLoader(train_dataset,batch_size=42,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=42)

### Deep Learning

In [43]:
# Build your ANN Model
class ANN(nn.Module):                              # Define ANN class

    def __init__(self):                            # Constructor
        super(ANN, self).__init__()                # Initialize parent class

        self.model = nn.Sequential(                # Combine layers sequentially

            nn.Linear(X_train.shape[1], 6),        # Input layer → 6 neurons
            nn.ReLU(),                             # Activation function

            nn.Linear(6, 6),                       # Hidden layer → 6 neurons
            nn.ReLU(),                             # Activation function

            nn.Linear(6, 1)                        # Output layer → 1 neuron
        )

    def forward(self, x):                          # Forward propagation
        return self.model(x)                       # Pass input through ANN

In [45]:
import torch.optim as optim
model=ANN()

#loss ,optimizer
crietrion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())